<a href="https://colab.research.google.com/github/Sabra7/Ai-usnig-py/blob/main/Lab2_MohmmedAlshaharani_445801999.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Intelligent Agents

**Introduction to Artificial Intelligence · 361CCS-3 / 3251CCS-2**
King Khalid University · College of Computer Science, Abha
Instructor: Mansour Al Ghanim

---

In the lecture we said an agent is a function from percept sequences to actions, and that the
*environment* determines which architecture you can get away with. This lab makes both claims
concrete: you will build the environment, drop different agents into it, and score them.

Everything here is plain Python. No libraries to install, no magic — you will see every line
of the agent.

### What you will be able to do after this lab

1. Write an agent as a function from percept to action.
2. Build a simulator that runs an agent in an environment and scores it.
3. Show by experiment where a simple reflex agent fails.
4. Fix it by giving the agent internal state, and measure the difference.
5. Classify an environment on the seven dimensions from the lecture.

**Time needed:** about 90 minutes
**Prerequisite:** Lab 1
**Submission:** see *Exercises* at the end.

> ⚠️ **First:** click **File → Save a copy in Drive**, then rename it
> `Lab2_YourName_YourID`. Nothing is saved until you do.

---
## Part 1 — The vacuum world

Two squares, A and B. Each is clean or dirty. The agent is in one of them and perceives only
`(location, status)` — it cannot see the other square.

This is the smallest environment that is still interesting, and it is the one AIMA uses.

In [ ]:
import random

class VacuumWorld:
    """Two squares. The agent sees only the square it is standing in."""

    def __init__(self, dirt=("A", "B"), start="A"):
        self.status = {"A": "Clean", "B": "Clean"}
        for sq in dirt:
            self.status[sq] = "Dirty"
        self.location = start
        self.score = 0
        self.log = []

    def percept(self):
        return (self.location, self.status[self.location])

    def step(self, action):
        # one point for every clean square, at every time step
        if action == "Suck":
            self.status[self.location] = "Clean"
        elif action == "Left":
            self.location = "A"
        elif action == "Right":
            self.location = "B"
        elif action == "NoOp":
            pass
        else:
            raise ValueError(f"unknown action: {action}")
        self.score += sum(1 for v in self.status.values() if v == "Clean")
        self.log.append((self.location, dict(self.status), action))

def run(agent, world, steps=10):
    for _ in range(steps):
        world.step(agent(world.percept()))
    return world.score

print("A world starts like this:", VacuumWorld().percept())

A world starts like this: ('A', 'Dirty')


### The agent is just a function

`percept in, action out`. That is the whole interface. Anything obeying it is an agent.

In [ ]:
def reflex_agent(percept):
    """Simple reflex agent: acts on the current percept alone."""
    location, status = percept
    if status == "Dirty":
        return "Suck"
    return "Right" if location == "A" else "Left"

w = VacuumWorld(dirt=("A", "B"), start="A")
print("score over 10 steps:", run(reflex_agent, w, steps=10))
for t, (loc, st, act) in enumerate(w.log[:6], start=1):
    print(f"  t={t}  at {loc}  world={st}  action={act}")

score over 10 steps: 18
  t=1  at A  world={'A': 'Clean', 'B': 'Dirty'}  action=Suck
  t=2  at B  world={'A': 'Clean', 'B': 'Dirty'}  action=Right
  t=3  at B  world={'A': 'Clean', 'B': 'Clean'}  action=Suck
  t=4  at A  world={'A': 'Clean', 'B': 'Clean'}  action=Left
  t=5  at B  world={'A': 'Clean', 'B': 'Clean'}  action=Right
  t=6  at A  world={'A': 'Clean', 'B': 'Clean'}  action=Left


---
## Part 2 — Where the simple reflex agent fails

Run it on a world that is **already clean**. Watch what it does.

In [ ]:
w = VacuumWorld(dirt=(), start="A")           # nothing to clean
print("score:", run(reflex_agent, w, steps=8))
print("actions:", [a for (_, _, a) in w.log])

score: 16
actions: ['Right', 'Left', 'Right', 'Left', 'Right', 'Left', 'Right', 'Left']


It paces between the two squares forever. Under our performance measure — one point per clean
square per time step — that is not catastrophic, but under a measure that **penalises movement**
it would be. And the agent has no way to notice, because it has no memory.

> **This is the lecture's point, demonstrated.** With no internal state, the agent cannot
> represent the fact "I have already checked both squares." A percept of `("A", "Clean")` looks
> identical the first time and the hundredth.

In [ ]:
def scored(agent, dirt, start="A", steps=10, move_penalty=1):
    """Same simulation, but now every movement costs a point."""
    w = VacuumWorld(dirt=dirt, start=start)
    for _ in range(steps):
        a = agent(w.percept())
        w.step(a)
        if a in ("Left", "Right"):
            w.score -= move_penalty
    return w.score

print("reflex, clean world, movement penalised:", scored(reflex_agent, ()))

reflex, clean world, movement penalised: 10


---
## Part 3 — Adding internal state

A **model-based reflex agent** keeps a model of the part of the world it cannot see. Here that
model is tiny: which squares have I confirmed clean?

Notice the agent is now a small object rather than a plain function, because it must remember
something between calls. It still obeys the same interface: percept in, action out.

In [ ]:
class ModelBasedAgent:
    """Remembers what it has seen, so it can stop when the job is done."""

    def __init__(self):
        self.model = {"A": None, "B": None}    # None = unknown

    def __call__(self, percept):
        location, status = percept
        self.model[location] = status          # update internal state
        if status == "Dirty":
            self.model[location] = "Clean"     # we are about to clean it
            return "Suck"
        if all(v == "Clean" for v in self.model.values()):
            return "NoOp"                      # job finished - stop moving
        return "Right" if location == "A" else "Left"

print("model-based, clean world, movement penalised:",
      scored(ModelBasedAgent(), ()))
print("model-based, both dirty,  movement penalised:",
      scored(ModelBasedAgent(), ("A", "B")))
print("reflex,       both dirty,  movement penalised:",
      scored(reflex_agent, ("A", "B")))

model-based, clean world, movement penalised: 19
model-based, both dirty,  movement penalised: 17
reflex,       both dirty,  movement penalised: 10


The model-based agent stops once it knows both squares are clean. The reflex agent cannot,
because "I already checked B" is not in its percept.

> **The general lesson.** Memory is not a nice extra. It is what a *partially observable*
> environment forces on you. That is the first row of the environment table from the lecture.

---
## Part 4 — Comparing agents properly

One run tells you very little. Average over many random worlds.

In [ ]:
def evaluate(make_agent, trials=200, steps=10, seed=0):
    """Average score over randomly generated worlds."""
    rng = random.Random(seed)
    total = 0
    for _ in range(trials):
        dirt = tuple(sq for sq in ("A", "B") if rng.random() < 0.5)
        start = rng.choice(["A", "B"])
        total += scored(make_agent(), dirt, start=start, steps=steps)
    return total / trials

print(f"simple reflex : {evaluate(lambda: reflex_agent):.2f}")
print(f"model-based   : {evaluate(lambda: ModelBasedAgent()):.2f}")

simple reflex : 10.27
model-based   : 18.20


Note `make_agent` is a *factory* — it builds a fresh agent for each trial. If we reused one
`ModelBasedAgent`, it would carry stale memory from the previous world into the next one. That
is a real bug, and an easy one to write.

---
## Part 5 — PEAS, in code

The lecture asked you to write PEAS descriptions on paper. Here is the same thing as a data
structure, for a system you already use.

In [ ]:
PEAS = {
    "task": "Crowd monitoring at the Grand Mosque (Baseer)",
    "performance": ["density kept below the danger threshold",
                    "early warning, not late alarm",
                    "minimal disruption to pilgrims"],
    "environment":  ["two million agents", "fixed ritual deadlines",
                     "a few square kilometres", "gates and access routes"],
    "actuators":    ["alert to operators", "gate and flow control",
                     "reroute message pushed through Nusuk"],
    "sensors":      ["cameras across the holy sites", "thermal imaging",
                     "drones", "gate counters"],
}

for k, v in PEAS.items():
    if k == "task":
        print(v); print("-" * len(v))
    else:
        print(f"{k.capitalize():<13}: " + "; ".join(v))

Crowd monitoring at the Grand Mosque (Baseer)
---------------------------------------------
Performance  : density kept below the danger threshold; early warning, not late alarm; minimal disruption to pilgrims
Environment  : two million agents; fixed ritual deadlines; a few square kilometres; gates and access routes
Actuators    : alert to operators; gate and flow control; reroute message pushed through Nusuk
Sensors      : cameras across the holy sites; thermal imaging; drones; gate counters


---
## Part 6 — Classifying the environment

The seven dimensions from the lecture, as a checklist you can apply to any task.

In [ ]:
DIMENSIONS = ["Fully observable?", "Single-agent?", "Deterministic?", "Episodic?",
              "Static?", "Discrete?", "Known?"]

def classify(name, answers):
    print(f"\n{name}")
    for d, a in zip(DIMENSIONS, answers):
        print(f"   {d:<20} {a}")

classify("Vacuum world (our simulator)",
         ["No", "Yes", "Yes", "No", "Yes", "Yes", "Yes"])
classify("Baseer, crowd monitoring",
         ["No", "No", "No", "No", "No", "No", "Partly"])


Vacuum world (our simulator)
   Fully observable?    No
   Single-agent?        Yes
   Deterministic?       Yes
   Episodic?            No
   Static?              Yes
   Discrete?            Yes
   Known?               Yes

Baseer, crowd monitoring
   Fully observable?    No
   Single-agent?        No
   Deterministic?       No
   Episodic?            No
   Static?              No
   Discrete?            No
   Known?               Partly


The vacuum world is **partially observable** — the agent cannot see the other square. That single
answer is what made memory necessary in Part 3. Everything else about the design followed from it.

---
## Exercises

Complete these in this notebook, in the cells provided.

### Exercise 1 — A new performance measure

Modify `scored` so that a movement costs **3** points instead of 1. Re-run `evaluate` for both agents. Does the ranking change? Answer in a text cell, in one sentence.

In [ ]:
def scored(agent, dirt, start="A", steps=10, move_penalty=3): #i only changeed the number here in move_penalty
    """Same simulation, but now every movement costs 3 points."""
    w = VacuumWorld(dirt=dirt, start=start)
    for _ in range(steps):
        a = agent(w.percept())
        w.step(a)
        if a in ("Left", "Right"):
            w.score -= move_penalty
    return w.score

In [ ]:
def evaluate(make_agent, trials=200, steps=10, seed=0):
    """Average score over randomly generated worlds."""
    rng = random.Random(seed)
    total = 0
    for _ in range(trials):
        dirt = tuple(sq for sq in ("A", "B") if rng.random() < 0.5)
        start = rng.choice(["A", "B"])
        total += scored(make_agent(), dirt, start=start, steps=steps)
    return total / trials

print(f"simple reflex : {evaluate(lambda: reflex_agent):.2f}")
print(f"model-based   : {evaluate(lambda: ModelBasedAgent()):.2f}")

simple reflex : -7.59
model-based   : 16.20


yes the ranking changed

from **simple reflex : 10.27**, **model-based   : 18.20**

to **simple reflex : -7.59, model-based   : 16.20**

### Exercise 2 — A three-square world

Extend `VacuumWorld` to squares A, B and C in a row (from A you can only reach B; from B you can reach A or C). Write a reflex agent for it. Then explain in a text cell why a *simple* reflex agent now has a problem it did not have with two squares.

In [ ]:
import random

class VacuumWorld:
    """three squares. The agent sees only the square it is standing in."""

    def __init__(self, dirt=("A", "B", "C"), start="A"):
        self.status = {"A": "Clean", "B": "Clean", "C":"Clean"}
        for sq in dirt: self.status[sq] = "Dirty"
        self.location = start
        self.score = 0
        self.log = []

    def percept(self):
        return (self.location, self.status[self.location])

    def step(self, action):
        # one point for every clean square, at every time step
        if action == "Suck":
            self.status[self.location] = "Clean"

        elif action == "Left":
            if self.location == "B": self.location = "A"
            elif self.location == "C": self.location = "B"

        elif action == "Right":
            if self.location == "B": self.location = "C"
            elif self.location == "A": self.location = "B"

        elif action == "NoOp":
            pass

        else:
            raise ValueError(f"unknown action: {action}")
        self.score += sum(1 for v in self.status.values() if v == "Clean")
        self.log.append((self.location, dict(self.status), action))

def run(agent, world, steps=10):
    for _ in range(steps):
        world.step(agent(world.percept()))
    return world.score

print("A world starts like this:", VacuumWorld().percept())

A world starts like this: ('A', 'Dirty')


In [ ]:
def reflex_agent_3(percept):
    location, status = percept
    if status == "Dirty":
        return "Suck"
    if location == "A":
        return "Right"
    if location == "C":
        return "Left"
    return "Right"

w = VacuumWorld(dirt=("A",), start="B")
for _ in range(8):
    w.step(reflex_agent_3(w.percept()))
print("visited:", [loc for (loc, _, _) in w.log])
print("final  :", w.status)

visited: ['C', 'B', 'C', 'B', 'C', 'B', 'C', 'B']
final  : {'A': 'Dirty', 'B': 'Clean', 'C': 'Clean'}


okay so when its 3 squares, at B the agent dont know wehre to go, on the right or on the left, but in 2 squares we only had one exit

### Exercise 3 — A model-based agent for three squares

Write a model-based agent for your three-square world that stops once all three are known clean. Compare its average score with the reflex agent over 200 random worlds.

In [ ]:
class ModelBasedAgent:
    """Remembers what it has seen, so it can stop when the job is done."""

    def __init__(self):
        self.model = {"A": None, "B": None, "C": None}    # None = unknown

    def __call__(self, percept):
        location, status = percept
        self.model[location] = status          # update internal state
        if status == "Dirty":
            self.model[location] = "Clean"     # we are about to clean it
            return "Suck"
        if all(v == "Clean" for v in self.model.values()):
            return "NoOp"                      # job finished - stop moving

        # if not< go to another
        if location == "A": return "Right"
        if location == "C": return "Left"
        # when he at B he will use the Memmory
        if self.model["A"] != "Clean": return "Left"
        return "Right"

print("model-based, clean world, movement penalised:",
      scored(ModelBasedAgent(), ()))
print("model-based, both dirty,  movement penalised:",
      scored(ModelBasedAgent(), ("A", "B","C")))
print("reflex,      both dirty,  movement penalised:",
      scored(reflex_agent, ("A", "B","C")))

model-based, clean world, movement penalised: 24
model-based, both dirty,  movement penalised: 18
reflex,      both dirty,  movement penalised: -6


In [ ]:
def evaluate(make_agent, trials=200, steps=10, seed=0):
    """Average score over randomly generated worlds."""
    rng = random.Random(seed)
    total = 0
    for _ in range(trials):
        dirt = tuple(sq for sq in ("A", "B", "C") if rng.random() < 0.5)
        start = rng.choice(["A", "B", "C"])
        total += scored(make_agent(), dirt, start=start, steps=steps)
    return total / trials

print(f"simple reflex : {evaluate(lambda: reflex_agent_3):.2f}")
print(f"model-based   : {evaluate(lambda: ModelBasedAgent()):.2f}")

simple reflex : -1.08
model-based   : 20.34


The model-based agent has a memmory so he know where to go, and sops when all clean

### Exercise 4 — PEAS and environment type

Choose one: an automatic Hajj-permit checker · a robot inspecting solar panels at Sakaka · an Arabic speech-to-text service. Write its PEAS as a Python dict like the one in Part 5, then classify it on all seven dimensions and name the **simplest** agent architecture that could work. Justify the architecture in a text cell.

In [ ]:
PEAS_HAJJ = {
    "task": "Automatic Hajj-permit checker (at a checkpoint gate)",
    "performance": ["no invalid permit accepted, no valid permit rejected",
                    "fast decision per pilgrim",
                    "clear reason shown for every rejection"],
    "environment": ["checkpoint gates and entry points",
                    "permit database and pilgrim records",
                    "pilgrims presenting a permit (QR code / ID)",
                    "fixed Hajj season dates"],
    "actuators":   ["open / keep closed the gate",
                    "display accept or reject message with the reason",
                    "alert a human officer for doubtful cases"],
    "sensors":     ["QR / barcode scanner", "ID or passport reader",
                    "camera for face matching (optional)",
                    "link to the permit database"],
}

for k, v in PEAS_HAJJ.items():
    if k == "task":
        print(v); print("-" * len(v))
    else:
        print(f"{k.capitalize():<13}: " + "; ".join(v))

Automatic Hajj-permit checker (at a checkpoint gate)
----------------------------------------------------
Performance  : no invalid permit accepted, no valid permit rejected; fast decision per pilgrim; clear reason shown for every rejection
Environment  : checkpoint gates and entry points; permit database and pilgrim records; pilgrims presenting a permit (QR code / ID); fixed Hajj season dates
Actuators    : open / keep closed the gate; display accept or reject message with the reason; alert a human officer for doubtful cases
Sensors      : QR / barcode scanner; ID or passport reader; camera for face matching (optional); link to the permit database


In [ ]:
classify("Hajj-permit checker",
         ["Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes"])


Hajj-permit checker
   Fully observable?    Yes
   Single-agent?        Yes
   Deterministic?       Yes
   Episodic?            Yes
   Static?              Yes
   Discrete?            Yes
   Known?               Yes


In [ ]:
print("\nSimplest architecture that could work: SIMPLE REFLEX AGENT")

def permit_agent(percept):
    """Condition-action rules on the current percept only; no memory needed."""
    permit_valid, season_ok, identity_matches = percept
    if permit_valid and season_ok and identity_matches:
        return "OpenGate"
    if not permit_valid:
        return "Reject: invalid permit"
    return "CallOfficer"

print(permit_agent((True, True, True)))
print(permit_agent((False, True, True)))
print(permit_agent((True, True, False)))


Simplest architecture that could work: SIMPLE REFLEX AGENT
OpenGate
Reject: invalid permit
CallOfficer


so for the checker a simple reflex agent is enough, because everything agent need (is the permit valid, is it the right season, does the ID match the person) the current percept and the database, so the environment is fully observable and also every pilgrim is a separate case and the environment is static, discrete and deterministic, so the agent doesn't need any memory.

---
## Before you submit

- [ ] **Runtime → Restart session and run all** completes with no errors
- [ ] Every exercise cell contains my own code
- [ ] Written answers are in **text cells**, not code comments
- [ ] Downloaded via **File → Download → Download .ipynb**
- [ ] Named `Lab2_YourName_YourID.ipynb` and uploaded to Blackboard

**Next lab:** Lab 3 — turning a real task into a search problem.